# Module 12 - 실습 1 솔루션: 체크포인팅 (MemorySaver)

In [ ]:
import sys
sys.path.insert(0, '../..')

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from common.fake_llm import FakeLLM

print("체크포인팅 솔루션 준비 완료")

## 체크포인터 종류 비교

| 체크포인터 | 저장소 | 영속성 | 사용 시나리오 |
|-----------|--------|--------|-------------|
| **MemorySaver** | 메모리 (dict) | 프로세스 종료 시 소멸 | 개발/테스트, 단일 세션 |
| **SqliteSaver** | SQLite 파일 | 파일 기반 영속 | 소규모 프로덕션, 단일 서버 |
| **PostgresSaver** | PostgreSQL | DB 기반 완전 영속 | 대규모 프로덕션, 분산 환경 |

> 이 실습에서는 MemorySaver를 사용합니다 (설치 없이 바로 사용 가능).

In [ ]:
class ProcessState(TypedDict):
    input_data: str
    step1_result: str | None
    step2_result: str | None
    step3_result: str | None
    current_step: str

In [ ]:
def step1_node(state: ProcessState) -> dict:
    """1단계: 데이터 수집."""
    print("  [Step 1] 데이터 수집 중...")
    return {
        "step1_result": f"수집 완료: {state['input_data']}",
        "current_step": "step1",
    }

def step2_node(state: ProcessState) -> dict:
    """2단계: 데이터 분석."""
    print("  [Step 2] 데이터 분석 중...")
    return {
        "step2_result": f"분석 완료: {state['step1_result']}",
        "current_step": "step2",
    }

def step3_node(state: ProcessState) -> dict:
    """3단계: 결과 생성."""
    print("  [Step 3] 결과 생성 중...")
    return {
        "step3_result": f"최종 결과: {state['step2_result']}",
        "current_step": "step3",
    }

# 그래프 구성
graph = StateGraph(ProcessState)
graph.add_node("step1", step1_node)
graph.add_node("step2", step2_node)
graph.add_node("step3", step3_node)

graph.set_entry_point("step1")
graph.add_edge("step1", "step2")
graph.add_edge("step2", "step3")
graph.add_edge("step3", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)
print("그래프 구성 완료")

In [ ]:
config = {"configurable": {"thread_id": "task-001"}}

initial_state = {
    "input_data": "사용자 로그 데이터",
    "step1_result": None,
    "step2_result": None,
    "step3_result": None,
    "current_step": "start",
}

print("그래프 실행 시작...")
result = app.invoke(initial_state, config=config)
print(f"\n최종 결과: {result['step3_result']}")

In [ ]:
# 체크포인트 상태 조회
state_snapshot = app.get_state(config)
print(f"마지막 단계: {state_snapshot.values.get('current_step', 'N/A')}")

In [ ]:
assert result is not None
assert result.get("step3_result") is not None
assert result.get("current_step") == "step3"
print("✅ 실습 3 완료! 3단계 파이프라인이 체크포인팅과 함께 실행되었습니다.")

In [ ]:
config2 = {"configurable": {"thread_id": "task-002"}}

initial_state2 = {
    "input_data": "다른 사용자의 데이터",
    "step1_result": None,
    "step2_result": None,
    "step3_result": None,
    "current_step": "start",
}

print("새 thread_id로 실행...")
result2 = app.invoke(initial_state2, config=config2)
print(f"결과: {result2['step3_result']}")

In [ ]:
assert result2 is not None
state1 = app.get_state({"configurable": {"thread_id": "task-001"}})
state2 = app.get_state({"configurable": {"thread_id": "task-002"}})
assert state1.values.get("input_data") == "사용자 로그 데이터"
assert state2.values.get("input_data") == "다른 사용자의 데이터"
print("✅ 실습 4 완료! 두 thread_id가 독립적으로 관리됩니다.")